# HW2 — Object detection on drone videos (VisDrone2019-VID)

All functions live in `hw2_detection.py`; this notebook only sets up the runs and shows the results.

| Section | Rubric item |
|---|---|
| 0. Setup | — |
| 1. Data understanding | Data Understanding and Preprocessing |
| 2. Preprocessing and validation protocol | Data Understanding and Preprocessing |
| 3. Faster R-CNN | Two-stage anchor-based |
| 4. RetinaNet | One-stage anchor-based |
| 5. FCOS | One-stage anchor-free |
| 6. RT-DETR | Query-based |
| 7. Comparison of all models | (input for Your pipeline) |
| 8. Baseline submission | valid submission from the best required model |
| 9. Your pipeline | *later* |

**Kaggle settings:** Accelerator = GPU T4 (recent PyTorch does not support the P100), Internet = On
(pretrained weights), competition data attached.
Every finished run is saved to `results/<run name>/`. If a run is already finished (also in an earlier
notebook version attached as input), it is loaded instead of retrained, so the notebook can be run
section by section within the 12-hour session limit.

## 0. Setup

In [ ]:
import os, sys, json, subprocess, importlib
from pathlib import Path

# ---- settings -----------------------------------------------------------------
REPO_URL = ""   # public GitHub repo with hw2_detection.py, e.g. "https://github.com/<user>/<repo>.git"
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"
DATA_DIR = None  # set a path to skip the automatic search / download
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RESULTS_DIR = WORK_DIR / "results"
CACHE_DIR = WORK_DIR / "cache"

# which sections train models in this session (finished runs are loaded either way)
RUN = {"faster_rcnn": True, "retinanet": True, "fcos": True, "rtdetr": True}
MODEL_ANALYSIS = True   # analyses that need a loaded model and a GPU (RPN recall, test predictions)
MAKE_SUBMISSION = True

# common protocol for all detectors (see section 2)
EPOCHS = 8
TRAIN_STEP = 10      # keep every 10th frame of each training clip
VAL_STEP = 5         # keep every 5th frame of each validation clip
MAX_MINUTES = None   # optional time cap per run (same budget for every model)
SEED = 42

# quick check that every model trains and evaluates (1 epoch, few frames, separate results folder)
SMOKE_TEST = False
if SMOKE_TEST:
    EPOCHS, TRAIN_STEP, VAL_STEP = 1, 200, 50
    RESULTS_DIR = WORK_DIR / "results_smoke"

In [ ]:
# install what is missing (Kaggle images normally have both)
def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

try:
    import pycocotools  # noqa: F401
except ImportError:
    pip_install("pycocotools")
try:
    from transformers import RTDetrForObjectDetection  # noqa: F401
except ImportError:
    pip_install("-U", "transformers")

In [ ]:
# make hw2_detection.py importable (cloned repo, notebook folder or an attached dataset)
if REPO_URL:
    repo_dir = WORK_DIR / "repo"
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)

search = [Path.cwd(), WORK_DIR / "repo", *sorted((WORK_DIR / "repo").glob("*")), *sorted(Path("/kaggle/input").glob("*"))]
module_dir = next((p for p in search if (p / "hw2_detection.py").exists()), None)
if module_dir is None:
    raise FileNotFoundError("hw2_detection.py not found: set REPO_URL or put the file next to the notebook")
sys.path.insert(0, str(module_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hw2_detection as hd
importlib.reload(hd)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("module:", module_dir / "hw2_detection.py")

In [ ]:
# competition data: attached input on Kaggle, otherwise downloaded with kagglehub
if DATA_DIR is None:
    try:
        import kagglehub
        DATA_DIR = kagglehub.competition_download(COMPETITION)
    except Exception as e:
        print("kagglehub failed:", repr(e), "-> searching /kaggle/input")
        DATA_DIR = "/kaggle/input"
DATA_ROOT = hd.find_data_root(DATA_DIR)
print("data root:", DATA_ROOT)
print(sorted(p.name for p in DATA_ROOT.iterdir()))

hd.seed_everything(SEED)
DEVICE = hd.get_device()
import torch
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU",
      "| torch", torch.__version__)
PREV_DIRS = hd.discover_prev_results("/kaggle/input")
print("finished runs from attached inputs:", PREV_DIRS)

## 1. Data understanding

The index has one row per frame: image path, size, boxes (pixels, `x1 y1 x2 y2`), labels (0–7) and ignored
regions, converted from the YOLO files. It is cached in `cache/` after the first run.

In [ ]:
train_full = hd.build_index(DATA_ROOT, "train", cache_dir=CACHE_DIR)
val_full = hd.build_index(DATA_ROOT, "val", cache_dir=CACHE_DIR)
test_df = hd.build_test_index(DATA_ROOT)
all_full = pd.concat([train_full, val_full], ignore_index=True)
objs = hd.boxes_table(all_full)

print("clips with different first/last frame size:", int((~all_full["size_consistent"]).sum()))
print("degenerate boxes (<1 px) removed:", int(all_full["n_degenerate"].sum()))
print(f"test: {test_df['clip'].nunique()} clips, {len(test_df)} frames")
hd.dataset_overview({"train": train_full, "val": val_full})

### 1.1 Classes

In [ ]:
display(hd.class_distribution(objs))
hd.plot_class_distribution(objs);

*Observations (fill in):* which classes dominate, which are rare, is val similar to train?

### 1.2 Object sizes and shapes

In [ ]:
display(hd.size_table(objs))
hd.plot_size_distribution(objs, min_size=800, max_size=1333);
hd.plot_aspect_ratios(objs);
hd.suggest_anchor_stats(objs, min_size=800, max_size=1333)

*Observations (fill in):* typical object size in pixels before / after resize, share of tiny objects,
aspect ratios per class. What does this mean for input resolution and anchor sizes?

### 1.3 Scene density and variation across sequences

In [ ]:
hd.plot_objects_per_frame(all_full);
clip_stats = hd.per_clip_stats(all_full, objs)
display(clip_stats)

*Observations (fill in):* range of objects per frame, how much clips differ (density, resolution,
object size, class mix), how the 7 validation clips compare with the training clips.

### 1.4 Ignored regions

In [ ]:
display(pd.DataFrame({"train": hd.ignore_stats(train_full), "val": hd.ignore_stats(val_full)}))

### 1.5 Temporal redundancy (for the frame subsampling step)

In [ ]:
redundancy = hd.temporal_redundancy(train_full, lags=(1, 2, 5, 10, 20, 40), n_clips=10, seed=SEED)
redundancy

`pixel_diff` — mean absolute grey-level difference between frame *t* and *t+lag*;
`box_persist` — share of objects at *t* that still have a same-class box with IoU > 0.5 at *t+lag*.

*Observations (fill in):* from which lag do frames stop being near-duplicates? Use this to justify `TRAIN_STEP`.

## 2. Preprocessing and validation protocol

### 2.1 Annotation conversion check
YOLO `cx cy w h` (normalised) → pixel `x1 y1 x2 y2`. Ground truth in class colours, ignored regions grey/hatched.

In [ ]:
hd.show_gt_samples(train_full, n=4, seed=SEED);
hd.show_gt_samples(val_full, n=2, seed=SEED);

In [ ]:
# compare the converted YOLO boxes with annotations_coco.json on random frames
coco_check = hd.crosscheck_with_coco(train_full, DATA_ROOT / "train" / "annotations_coco.json", n_frames=200, seed=SEED)
coco_check.describe().loc[["mean", "min", "max"]]

In [ ]:
# occlusion / truncation from the COCO file, used later in the failure analysis
try:
    val_full = hd.attach_coco_attributes(val_full, DATA_ROOT / "val" / "annotations_coco.json")
    print("val boxes matched to COCO attributes:", sum(int((o >= 0).sum()) for o in val_full["occlusion"]),
          "of", int(val_full["n_obj"].sum()))
except Exception as e:
    print("COCO attributes not attached:", repr(e))

**Ignored regions and non-target classes.** VisDrone "ignored regions" and "others" are given only as
ignored regions (no class). During training they are painted with the mean colour and GT boxes lying
more than 50 % inside them are dropped, so the model is neither rewarded nor punished there. During
evaluation, detections lying more than 50 % inside an ignored region are removed before scoring (unless
they match an annotated box), which reproduces "detections there are not penalised".

### 2.2 Split and frame subsampling
The organisers' `val/` clips are a sequence-level hold-out (checked below). All model comparisons use the
same training frames, the same validation frames, the same metric code and the same seed.

In [ ]:
print("clips in both train and val:", hd.check_clip_overlap(train_full, val_full) or "none")
train_df = hd.subsample_frames(train_full, TRAIN_STEP)
val_df = hd.subsample_frames(val_full, VAL_STEP)
print(f"train: {len(train_full)} -> {len(train_df)} frames | val: {len(val_full)} -> {len(val_df)} frames")
display(hd.class_distribution(hd.boxes_table(pd.concat([train_df, val_df], ignore_index=True))))

In [ ]:
# optional: clip-level folds (GroupKFold) for later cross-validation of the final pipeline
folds = hd.make_clip_folds(train_full, n_splits=5, seed=SEED)
folds.groupby("fold").agg(clips=("clip", "nunique"), frames=("clip", "size"), objects=("n_obj", "sum"))

### 2.3 Resizing, normalisation, augmentation
* **Resize** (inside every model, same rule): shorter side → 800 px, longer side ≤ 1333 px, aspect ratio kept.
* **Normalisation**: torchvision models use ImageNet mean/std internally; RT-DETR uses 0–1 inputs (as its
  image processor does).
* **Augmentation** (training only): horizontal flip p = 0.5. Colour jitter is available but off in the protocol.
* Reproducibility: fixed seed, seeded DataLoader generator, the same subsampled frames for all models.

In [ ]:
aug_ds = hd.DroneDetDataset(train_df, train=True, mask_ignore=True, hflip_p=0.5)
example = int(np.argmax(train_df["n_ignore"].to_numpy() > 0))  # a frame with ignored regions
hd.show_augmented(aug_ds, example, n=3, seed=SEED);

In [ ]:
# shared protocol: every experiment = these settings + the changes listed in its section
PROTOCOL = dict(epochs=EPOCHS, train_step=TRAIN_STEP, val_step=VAL_STEP, max_minutes=MAX_MINUTES, seed=SEED)
BASE = {arch: hd.make_config(arch, **PROTOCOL) for arch in hd.ARCH_DEFAULTS}
pd.Series({k: v for k, v in BASE["faster_rcnn"].items() if k in hd.COMMON_CFG}, dtype=object).to_frame("value")

In [ ]:
def experiments(arch, **variants):
    # {run name: config}; each variant = BASE[arch] + its changes
    return {name: hd.make_config(arch, **PROTOCOL, **changes) for name, changes in variants.items()}


def run_or_load(exps, train=True):
    # train (or load if finished) every experiment; returns {name: result}
    out = {}
    for name, cfg in exps.items():
        if train:
            out[name] = hd.run_experiment(name, cfg, train_df, val_df, RESULTS_DIR, PREV_DIRS, DEVICE)
        else:
            found = next((Path(d) / name for d in [RESULTS_DIR, *PREV_DIRS]
                          if (Path(d) / name / "metrics.json").exists()), None)
            if found is not None:
                out[name] = hd.load_run(found)
    return out


def run_dir(name):
    for d in [RESULTS_DIR, *PREV_DIRS]:
        if (Path(d) / name / "metrics.json").exists():
            return Path(d) / name
    raise FileNotFoundError(name)


def best_of(results):
    return max(results.values(), key=lambda r: r["mAP"]) if results else None


def changes_table(exps):
    diffs = {n: {k: str(v) for k, v in hd.config_diff(c, BASE[c["arch"]]).items()} for n, c in exps.items()}
    return pd.DataFrame(diffs).T.fillna("")

## 3. Two-stage anchor-based: Faster R-CNN

`fasterrcnn_resnet50_fpn_v2` (torchvision), COCO-pretrained, new classifier for 8 classes.
Experiments are cumulative (each one keeps the previous change):

1. `frcnn_base` — default settings.
2. `frcnn_props` — more RPN proposals (dense scenes with 100+ objects; default keeps 1000 at test time).
3. `frcnn_props_giou` — + GIoU box regression loss in the second stage instead of smooth-L1.

In [ ]:
PROPS = dict(rpn_pre_nms_top_n_train=3000, rpn_post_nms_top_n_train=3000,
             rpn_pre_nms_top_n_test=3000, rpn_post_nms_top_n_test=2000)
FRCNN_EXPS = experiments(
    "faster_rcnn",
    frcnn_base={},
    frcnn_props=PROPS,
    frcnn_props_giou={**PROPS, "box_loss": "giou"},
    # frcnn_props_roi={**PROPS, "box_batch_size_per_image": 1024, "box_positive_fraction": 0.5},  # optional
)
changes_table(FRCNN_EXPS)

In [ ]:
frcnn_res = run_or_load(FRCNN_EXPS, train=RUN["faster_rcnn"])
hd.results_table(list(frcnn_res.values())) if frcnn_res else None

In [ ]:
if frcnn_res:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    hd.plot_histories({n: r.get("history") for n, r in frcnn_res.items()}, ax=axes[0])
    hd.plot_train_losses({n: r.get("history") for n, r in frcnn_res.items()}, ax=axes[1])
    plt.show()
    display(hd.per_class_table(list(frcnn_res.values())))

### 3.1 What the two stages contribute
For each GT box: is it covered (IoU ≥ 0.5) by the top-N RPN proposals, and by the final detections?
`best_iou_*` shows how much the second stage refines localisation.

In [ ]:
FRCNN_BEST = best_of(frcnn_res)["name"] if frcnn_res else None
if FRCNN_BEST and MODEL_ANALYSIS and torch.cuda.is_available():
    frcnn_model = hd.load_model_from_run(run_dir(FRCNN_BEST), DEVICE)
    display(hd.rpn_proposal_recall(frcnn_model, val_df, DEVICE, n_images=100, top_ns=(100, 300, 1000, 2000)))
    del frcnn_model
    torch.cuda.empty_cache()

*Observations (fill in):* recall of proposals vs. final detections by object size; what does stage 2 add / lose?

### 3.2 Failure cases
Error types at score ≥ 0.3: missed objects, background false positives, wrong class, poor localisation, duplicates.
Green = correct, red = false positive, yellow dashed = missed, grey = ignored region.

In [ ]:
def failure_report(name, score_thr=0.3):
    preds = hd.load_val_preds(run_dir(name))
    gt_tab, pr_tab, conf = hd.analyze_failures(val_df, preds, score_thr=score_thr)
    for title, tab in hd.failure_summary(gt_tab, pr_tab).items():
        print("--", title)
        display(tab)
    print("-- class confusions (rows = GT, columns = prediction)")
    display(conf)
    return preds, gt_tab, pr_tab


if FRCNN_BEST:
    frcnn_preds, frcnn_gt_tab, frcnn_pr_tab = failure_report(FRCNN_BEST)

In [ ]:
if FRCNN_BEST:
    for title, rows in hd.pick_failure_frames(frcnn_gt_tab, frcnn_pr_tab, n_each=1).items():
        for row in rows:
            hd.plot_failure_frame(val_df, frcnn_preds, row, score_thr=0.3, title=title)
            plt.show()

*Failure cases (fill in, at least three):* for each frame above — what went wrong and the plausible cause
(object size, occlusion, density, similar classes, motion blur, ...), backed by the tables above.

## 4. One-stage anchor-based: RetinaNet

`retinanet_resnet50_fpn_v2` (torchvision), COCO-pretrained, same protocol.
First check how well anchor sets match the object sizes from section 1 (after resize).

In [ ]:
DEFAULT_ANCHORS = hd.retinanet_anchor_sizes((32, 64, 128, 256, 512))
SMALL_ANCHORS = hd.retinanet_anchor_sizes((16, 32, 64, 128, 256))   # chosen from section 1.2
RATIOS = (0.5, 1.0, 2.0)
flat = lambda sizes: [s for level in sizes for s in level]
print("default:", DEFAULT_ANCHORS)
print("small  :", SMALL_ANCHORS)
pd.concat({"default anchors": hd.anchor_coverage(objs, flat(DEFAULT_ANCHORS), RATIOS),
           "small anchors": hd.anchor_coverage(objs, flat(SMALL_ANCHORS), RATIOS)}, axis=1)

`covered_pct` = share of GT boxes whose best anchor shape reaches IoU ≥ 0.5 (the positive threshold),
centres aligned — an upper bound for the real assignment.

In [ ]:
RETINA_EXPS = experiments(
    "retinanet",
    retina_base={},
    retina_anchors={"anchor_base_sizes": (16, 32, 64, 128, 256)},
    retina_anchors_focal={"anchor_base_sizes": (16, 32, 64, 128, 256), "focal_gamma": 1.5},
)
changes_table(RETINA_EXPS)

In [ ]:
retina_res = run_or_load(RETINA_EXPS, train=RUN["retinanet"])
if retina_res:
    hd.plot_histories({n: r.get("history") for n, r in retina_res.items()})
    plt.show()
compare = list(retina_res.values()) + ([frcnn_res[FRCNN_BEST]] if FRCNN_BEST else [])
hd.results_table(compare) if compare else None

*Discussion (fill in):* what can go wrong without a proposal stage (e.g. extreme foreground/background
imbalance over ~100k anchors, no second refinement step) and how RetinaNet addresses it (focal loss,
IoU thresholds, anchor design). Initial vs tuned RetinaNet vs tuned Faster R-CNN.

## 5. One-stage anchor-free: FCOS

`FCOS ResNet-50-FPN` (torchvision), COCO-pretrained. Every feature-map point is a candidate; it is positive
for a box if it lies inside the box, near its centre (`center_sampling_radius × stride`) and the box size fits
the FPN level. The simulation below counts positive locations per GT box for different settings.

In [ ]:
assign = {}
for label, strides, radius in [("P3-P7, r=1.5 (default)", hd.fcos_strides(False), 1.5),
                               ("P3-P7, r=2.5", hd.fcos_strides(False), 2.5),
                               ("P2-P7, r=1.5", hd.fcos_strides(True), 1.5)]:
    tab, summary = hd.fcos_assignment_stats(train_df, strides=strides, center_radius=radius, n_images=300, seed=SEED)
    assign[label] = tab
    print(label, summary)
pd.concat(assign, axis=1)

In [ ]:
FCOS_EXPS = experiments(
    "fcos",
    fcos_base={},
    fcos_p2={"p2_level": True},                       # extra high-resolution level (stride 4)
    fcos_p2_ciou={"p2_level": True, "box_loss": "ciou"},
)
changes_table(FCOS_EXPS)

In [ ]:
fcos_res = run_or_load(FCOS_EXPS, train=RUN["fcos"])
if fcos_res:
    hd.plot_histories({n: r.get("history") for n, r in fcos_res.items()})
    plt.show()
compare = list(fcos_res.values()) + [r for r in (best_of(frcnn_res), best_of(retina_res)) if r]
hd.results_table(compare, cols=["mAP", "AP50", "AP_tiny", "AP_small", "AP_medium", "AP_large", "AR", "fps"]) if compare else None

*Observations (fill in):* search range and selected settings; effect of P2 / loss on tiny and small objects
and on dense frames; comparison with the tuned anchor-based models by object size.

## 6. Query-based: RT-DETR

`PekingU/rtdetr_r50vd` (Hugging Face), COCO-pretrained, same resize rule, same epochs. AdamW,
backbone learning rate × 0.1. The number of queries is the maximum number of objects the model can output
per image, so it is compared with the scene density first.

In [ ]:
print("objects per frame quantiles (train):")
print(train_full["n_obj"].quantile([0.5, 0.9, 0.99, 1.0]).round(0).to_dict())

In [ ]:
RTDETR_EXPS = experiments(
    "rtdetr",
    rtdetr_base={},
    rtdetr_q500={"num_queries": 500},
    rtdetr_q500_giou={"num_queries": 500, "matcher_giou_cost": 4.0, "weight_loss_giou": 4.0},
)
changes_table(RTDETR_EXPS)

In [ ]:
rtdetr_res = run_or_load(RTDETR_EXPS, train=RUN["rtdetr"])
hd.results_table(list(rtdetr_res.values())) if rtdetr_res else None

In [ ]:
CNN_BEST = best_of({**frcnn_res, **retina_res, **fcos_res})
if rtdetr_res and CNN_BEST:
    curves = {**{n: r.get("history") for n, r in rtdetr_res.items()}, CNN_BEST["name"]: CNN_BEST.get("history")}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    hd.plot_histories(curves, x="epoch", ax=axes[0], title="val mAP vs epoch")
    hd.plot_histories(curves, x="minutes", ax=axes[1], title="val mAP vs training time")
    plt.show()

In [ ]:
# small objects and dense scenes: RT-DETR vs the best CNN on the same frames
if rtdetr_res and CNN_BEST:
    RTDETR_BEST = best_of(rtdetr_res)["name"]
    dens = {}
    for name in (RTDETR_BEST, CNN_BEST["name"]):
        dens[name] = hd.ap_by_density(val_df, hd.load_val_preds(run_dir(name)), bins=(0, 50, 100, 200, np.inf))
    display(pd.concat(dens, axis=1))
    display(hd.results_table([rtdetr_res[RTDETR_BEST], CNN_BEST],
                             cols=["mAP", "AP50", "AP_tiny", "AP_small", "AP_medium", "AP_large", "AR", "fps"]))

*Observations (fill in):* how bipartite matching replaces NMS and what it changes in training; has the model
converged within the budget; small objects and dense scenes vs the CNN.

## 7. Comparison of all models

In [ ]:
ALL_RES = {**frcnn_res, **retina_res, **fcos_res, **rtdetr_res}
if ALL_RES:
    summary = hd.results_table(list(ALL_RES.values()))
    display(summary)
    display(hd.per_class_table(list(ALL_RES.values())))
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for arch, g in summary.groupby("arch"):
        ax.scatter(g["fps"], g["mAP"], label=arch)
        for name, r in g.iterrows():
            ax.annotate(name, (r["fps"], r["mAP"]), fontsize=7)
    ax.set_xlabel("images / s (batch 1)")
    ax.set_ylabel("val mAP")
    ax.set_title("Accuracy vs speed")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(RESULTS_DIR / "summary_required_models.csv")

## 8. Baseline submission
Predictions of the best required model (by validation mAP) on the test set — a valid submission to check the
format and beat the baseline early. It is not the final pipeline.

In [ ]:
if MAKE_SUBMISSION and ALL_RES and MODEL_ANALYSIS:
    best = best_of(ALL_RES)
    print("model:", best["name"], "| val mAP", round(best["mAP"], 4))
    model = hd.load_model_from_run(run_dir(best["name"]), DEVICE)
    with open(run_dir(best["name"]) / "config.json") as f:
        best_cfg = json.load(f)
    test_preds = hd.predict_df(model, test_df, best_cfg, DEVICE)
    sub_path = WORK_DIR / "submission.csv"
    hd.make_submission(test_df, test_preds, sub_path, max_det=500)
    print(hd.validate_submission(sub_path, DATA_ROOT / "sample_submission.csv"))
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. Your pipeline
*To be done later: data / inference strategy experiments and model experiments, built on the results above.*